# Speech audition (attended, free)

Runs one studio-cloudrun **model-run** job with `"where": "attended"` on this notebook's own runtime.
Nothing is priced or paid for by the kit: pick a GPU runtime yourself (Runtime > Change runtime type > T4 or L4) and watch it.

1. Set the kit commit below (the one the job was reviewed against).
2. Upload the job file and, for speech recipes, the approved reference voice.
3. Run. Download the zip at the end. Then disconnect the runtime.

Rules: the reference must be an original voice or one you are authorized to use (never an imitation of a real person).
The reference stays on this runtime: results never include it, and a result that is a byte copy of it fails the run.

In [ ]:
KIT_REPO = "https://github.com/GanglyPuma22/game-studio-kit.git"  # private: Colab asks for GitHub access, or upload a kit zip instead
KIT_COMMIT = ""  # full 40-character commit the job was planned against
assert len(KIT_COMMIT) == 40, "set KIT_COMMIT"
!rm -rf /content/kit && git clone -q "$KIT_REPO" /content/kit && git -C /content/kit checkout -q --detach "$KIT_COMMIT" && git -C /content/kit log --oneline -1

In [ ]:
# Upload job.json first, then the input files it lists (skip the second picker for casting jobs).
import json, os, shutil
from google.colab import files
os.makedirs("/content/inputs", exist_ok=True)
up = files.upload()
job_name = next(n for n in up if n.endswith(".json"))
shutil.move(job_name, "/content/job.json")
job = json.load(open("/content/job.json"))
print(job["id"], job["recipe"], "where:", job.get("where"), "max_units:", job.get("max_units"))
for item in job.get("inputs") or []:
    print("needs input:", item["path"])
if job.get("inputs"):
    for name in files.upload():
        shutil.move(name, os.path.join("/content/inputs", name))

In [ ]:
# Check before anything heavy: job fields, plan, inputs against their SHA-256.
!python3 /content/kit/scripts/studio.py cloudrun validate --job /content/job.json --inputs /content/inputs
!python3 /content/kit/scripts/studio.py cloudrun plan --job /content/job.json

In [ ]:
# Setup (pinned environment, pinned weights checked file by file), then the takes.
import time
RESULTS = f"/content/results-{job['id']}-{time.strftime('%Y%m%dT%H%M%S')}"  # fresh per run
!bash /content/kit/skills/studio-cloudrun/scripts/attended.sh /content/job.json /content/inputs "$RESULTS"

In [ ]:
from google.colab import files
files.download(RESULTS + ".zip")

When the download finishes: Runtime > Disconnect and delete runtime. The reference voice goes with it.